# Downloading 13 years of data and storing in "nifty50_13years.csv" file

In [1]:
!pip install tensorflow==2.10

In [2]:
import tensorflow as tf
print(tf.__version__)

2.10.0


In [3]:
!pip install tensorflow


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import joblib

from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Layer
from tensorflow.keras.optimizers import Adam
import tensorflow as tf



def download_data():

    print("Downloading NIFTY data...")

    nifty = yf.download("^NSEI", period="max", interval="1d")

    nifty = nifty[['Open', 'High', 'Low', 'Close']]

    nifty.dropna(inplace=True)

    nifty.to_csv("data/nifty_ohlc.csv")

    return nifty


data = download_data()


print("Applying preprocessing...")

# Log transform
log_data = np.log(data)

# Standardization
scaler = StandardScaler()
scaled_data = scaler.fit_transform(log_data)

# Save scaler
joblib.dump(scaler, "model/scaler.pkl")

print("Scaler saved.")



WINDOW = 30

X = []
y = []

for i in range(WINDOW, len(scaled_data)):
    X.append(scaled_data[i-WINDOW:i])
    y.append(scaled_data[i, 3])  # Close column

X = np.array(X)
y = np.array(y)

print("Windowed data shape:", X.shape)



class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="normal"
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros"
        )
        super().build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)


print("Building model...")

input_layer = Input(shape=(30, 4))

lstm_out = LSTM(50, return_sequences=True)(input_layer)

attention_out = AttentionLayer()(lstm_out)

dropout = Dropout(0.2)(attention_out)

output = Dense(1)(dropout)

model = Model(inputs=input_layer, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

model.summary()



print("Training model...")

history = model.fit(
    X, y,
    epochs=100,
    batch_size=64,
    validation_split=0.1
)


model.save("model/lstm_attention.h5")

print("Model saved successfully.")


[*********************100%***********************]  1 of 1 completed


Applying preprocessing...
Scaler saved.
Windowed data shape: (4505, 30, 4)
Building model...
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 30, 4)]           0         
                                                                 
 lstm (LSTM)                 (None, 30, 50)            11000     
                                                                 
 attention_layer (AttentionL  (None, 50)               80        
 ayer)                                                           
                                                                 
 dropout (Dropout)           (None, 50)                0         
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 11,131
Trainable par